# 🎮 Práctica 08: 3D Scatter Plot con Sprites de Pokémons

---

| Campo | Detalle |
|-------|--------|
| **Estudiante** | Francisco Garcia Garcia |
| **Matrícula** | 230758 |
| **Grupo** | 9°A - IDGS |
| **Materia** | Extracción de Conocimiento en Bases de Datos (ECBD) |
| **Fecha** | 06 de Agosto de 2026 |

---

## Objetivo

Construir un **Scatter Plot 3D interactivo** utilizando **Plotly** que permita visualizar y explorar las estadísticas base de los Pokémon (generaciones 1 a 9), diferenciando visualmente cada tipo principal mediante colores, integrando **sprites oficiales** en la información emergente, y aplicando filtros interactivos por generación, tipo y rango de estadísticas para identificar patrones, agrupaciones y valores atípicos.

## Fuentes de Datos

- **Dataset:** [lgreski/pokemonData](https://github.com/lgreski/pokemonData) — 1,215 Pokémon con estadísticas base (Gen 1–9), cortesía de pokemondb.net
- **Sprites:** [PokeAPI Sprites](https://github.com/PokeAPI/sprites) — Imágenes oficiales de cada Pokémon
- **Referencia:** [Unsupervised Learning: K-Means EDA (Kaggle)](https://www.kaggle.com/code/tanmay111999/unsupervised-learning-3-6-clusters-k-means-eda) — Técnicas de visualización 3D con Plotly

---

## 1. Importación de Librerías

Importamos las librerías necesarias para la manipulación, análisis y visualización de datos:
- **Pandas:** Manipulación y análisis de DataFrames
- **NumPy:** Operaciones numéricas y cálculos estadísticos
- **Plotly:** Visualizaciones 3D interactivas (graph_objects y express)
- **IPython.display:** Visualización enriquecida en Jupyter

In [1]:
# ============================================================
# Importación de Librerías
# ============================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML, Image
import warnings

# Configuración general
warnings.filterwarnings('ignore')
pio.templates.default = 'plotly_dark'
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print('✅ Librerías importadas correctamente')
print(f'   📦 Pandas:  {pd.__version__}')
print(f'   📦 NumPy:   {np.__version__}')
print(f'   📦 Plotly:  {pio.__version__ if hasattr(pio, "__version__") else "disponible"}')

✅ Librerías importadas correctamente
   📦 Pandas:  2.2.2
   📦 NumPy:   2.3.1
   📦 Plotly:  disponible


---

## 2. Carga del Dataset

El dataset proviene del repositorio [lgreski/pokemonData](https://github.com/lgreski/pokemonData) en GitHub, que contiene estadísticas básicas de **1,025 Pokémon únicos** (con formas alternativas el total supera los 1,200 registros) de las **generaciones 1 a 9**, recopiladas de [pokemondb.net](https://pokemondb.net).

**Columnas del dataset:**
- `ID` — Número del Pokédex Nacional
- `Name` — Nombre del Pokémon
- `Form` — Forma/variante (Mega, Alolan, Galarian, etc.)
- `Type1`, `Type2` — Tipos principal y secundario
- `Total` — Suma total de estadísticas base
- `HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, `Speed` — Estadísticas individuales
- `Generation` — Generación a la que pertenece

In [2]:
# ============================================================
# Carga del Dataset desde el repositorio de GitHub
# ============================================================
url_dataset = 'https://raw.githubusercontent.com/lgreski/pokemonData/master/Pokemon.csv'

# También se puede cargar localmente:
# df = pd.read_csv('Pokemon.csv')

df = pd.read_csv(url_dataset)

print(f'✅ Dataset cargado exitosamente')
print(f'   📊 Registros: {df.shape[0]}')
print(f'   📋 Columnas:  {df.shape[1]}')
print(f'   📁 Fuente:    lgreski/pokemonData (GitHub)')

✅ Dataset cargado exitosamente
   📊 Registros: 1215
   📋 Columnas:  13
   📁 Fuente:    lgreski/pokemonData (GitHub)


---

## 3. Inspección Inicial del Dataset

Realizamos una inspección completa del dataset utilizando las funciones estándar de Pandas para comprender su estructura, tipos de datos y distribución estadística.

### 3.1 Primeros registros (`head()`)

In [3]:
# ============================================================
# Primeros 10 registros del dataset
# ============================================================
df.head(10)

,ID,Name,Form,Type1,Type2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
0,1,Bulbasaur,,Grass,Poison,318,45,49,49,65,65,45,1
1,2,Ivysaur,,Grass,Poison,405,60,62,63,80,80,60,1
2,3,Venusaur,,Grass,Poison,525,80,82,83,100,100,80,1
3,4,Charmander,,Fire,,309,39,52,43,60,50,65,1
4,5,Charmeleon,,Fire,,405,58,64,58,80,65,80,1
5,6,Charizard,,Fire,Flying,534,78,84,78,109,85,100,1
6,7,Squirtle,,Water,,314,44,48,65,50,64,43,1
7,8,Wartortle,,Water,,405,59,63,80,65,80,58,1
8,9,Blastoise,,Water,,530,79,83,100,85,105,78,1
9,10,Caterpie,,Bug,,195,45,30,35,20,20,45,1


### 3.2 Dimensiones del Dataset (`shape`)

In [4]:
# ============================================================
# Dimensiones del dataset
# ============================================================
print(f'📐 Dimensiones del dataset:')
print(f'   Filas (registros):  {df.shape[0]}')
print(f'   Columnas (campos):  {df.shape[1]}')
print(f'\n📋 Nombres de columnas:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i:2d}. {col}')

📐 Dimensiones del dataset:
   Filas (registros):  1215
   Columnas (campos):  13

📋 Nombres de columnas:
    1. ID
    2. Name
    3. Form
    4. Type1
    5. Type2
    6. Total
    7. HP
    8. Attack
    9. Defense
   10. Sp. Atk
   11. Sp. Def
   12. Speed
   13. Generation


### 3.3 Información del Dataset (`info()`)

In [5]:
# ============================================================
# Información detallada del dataset
# ============================================================
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1215 entries, 0 to 1214
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          1215 non-null   int64 
 1   Name        1215 non-null   object
 2   Form        1215 non-null   object
 3   Type1       1215 non-null   object
 4   Type2       1215 non-null   object
 5   Total       1215 non-null   int64 
 6   HP          1215 non-null   int64 
 7   Attack      1215 non-null   int64 
 8   Defense     1215 non-null   int64 
 9   Sp. Atk     1215 non-null   int64 
 10  Sp. Def     1215 non-null   int64 
 11  Speed       1215 non-null   int64 
 12  Generation  1215 non-null   int64 
dtypes: int64(9), object(4)
memory usage: 123.5+ KB


### 3.4 Estadísticas Descriptivas (`describe()`)

In [6]:
# ============================================================
# Estadísticas descriptivas de columnas numéricas
# ============================================================
df.describe().round(2)

,ID,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
count,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00,1215.00
mean,501.74,443.10,71.24,81.15,75.01,73.22,72.44,70.03,5.06
std,298.98,121.19,26.93,32.04,30.74,32.76,27.58,30.16,2.60
min,1.00,175.00,1.00,5.00,5.00,10.00,20.00,5.00,1.00
25%,240.50,332.00,52.00,57.00,52.00,50.00,51.00,45.00,3.00
50%,495.00,465.00,70.00,80.00,70.00,65.00,70.00,68.00,5.00
75%,753.50,521.00,85.00,100.00,91.00,95.00,90.00,91.00,7.00
max,1025.00,1125.00,255.00,190.00,250.00,194.00,250.00,200.00,9.00


In [7]:
# ============================================================
# Estadísticas descriptivas de columnas categóricas
# ============================================================
df.describe(include='object')

,Name,Form,Type1,Type2
count,1215,1215,1215,1215
unique,1026,207,18,19
top,Rotom,,Water,
freq,6,985,150,546


---

## 4. Limpieza y Normalización de Datos

Procedemos a limpiar y normalizar el dataset para garantizar la calidad de los datos antes del análisis:
1. Renombrar columnas a formato `snake_case` para consistencia
2. Limpiar espacios en blanco en valores categóricos
3. Identificar y tratar valores nulos
4. Detectar y eliminar registros duplicados

### 4.1 Estado ANTES de la Limpieza

In [8]:
# ============================================================
# Estado del dataset ANTES de la limpieza
# ============================================================
print('=' * 60)
print('📋 ESTADO ANTES DE LA LIMPIEZA')
print('=' * 60)

# Valores nulos
print('\n🔍 Valores nulos por columna:')
nulos_antes = df.isnull().sum()
print(nulos_antes[nulos_antes > 0] if nulos_antes.sum() > 0 else '   No se encontraron valores nulos explícitos')

# Valores en blanco o espacios en Type2
print(f'\n🔍 Valores en blanco/espacios en Type1: {(df["Type1"].str.strip() == "").sum()}')
print(f'🔍 Valores en blanco/espacios en Type2: {(df["Type2"].str.strip() == "").sum()}')

# Duplicados
print(f'\n🔍 Registros duplicados: {df.duplicated().sum()}')

# Columnas actuales
print(f'\n🔍 Nombres de columnas originales: {list(df.columns)}')

# Tipos únicos
print(f'\n🔍 Tipos únicos en Type1: {sorted(df["Type1"].str.strip().unique())}')

📋 ESTADO ANTES DE LA LIMPIEZA

🔍 Valores nulos por columna:
   No se encontraron valores nulos explícitos

🔍 Valores en blanco/espacios en Type1: 0
🔍 Valores en blanco/espacios en Type2: 546

🔍 Registros duplicados: 0

🔍 Nombres de columnas originales: ['ID', 'Name', 'Form', 'Type1', 'Type2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation']

🔍 Tipos únicos en Type1: ['Bug', 'Dark', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Flying', 'Ghost', 'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Steel', 'Water']


### 4.2 Proceso de Limpieza y Normalización

In [9]:
# ============================================================
# Proceso de limpieza y normalización
# ============================================================

# 1. Renombrar columnas a snake_case
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('.', '', regex=False)
)
print('✅ 1. Columnas renombradas a snake_case')
print(f'   {list(df.columns)}')

# 2. Limpiar espacios en blanco en columnas de texto
for col in ['name', 'form', 'type1', 'type2']:
    df[col] = df[col].str.strip()
print('\n✅ 2. Espacios en blanco eliminados de columnas de texto')

# 3. Reemplazar cadenas vacías en type2 por 'None' (Pokémon de un solo tipo)
df['type2'] = df['type2'].replace('', 'None')
print(f'\n✅ 3. Valores vacíos en type2 reemplazados por "None"')
print(f'   Pokémon con un solo tipo: {(df["type2"] == "None").sum()}')
print(f'   Pokémon con dos tipos:    {(df["type2"] != "None").sum()}')

# 4. Reemplazar cadenas vacías en form por 'Standard'
df['form'] = df['form'].replace('', 'Standard')
print(f'\n✅ 4. Valores vacíos en form reemplazados por "Standard"')

# 5. Verificar y eliminar duplicados
duplicados = df.duplicated().sum()
if duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'\n✅ 5. Se eliminaron {duplicados} registros duplicados')
else:
    print(f'\n✅ 5. No se encontraron registros duplicados')

# 6. Verificar tipos de datos numéricos
cols_numericas = ['id', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation']
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(f'\n✅ 6. Tipos de datos numéricos verificados')

✅ 1. Columnas renombradas a snake_case
   ['id', 'name', 'form', 'type1', 'type2', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation']

✅ 2. Espacios en blanco eliminados de columnas de texto

✅ 3. Valores vacíos en type2 reemplazados por "None"
   Pokémon con un solo tipo: 546
   Pokémon con dos tipos:    669

✅ 4. Valores vacíos en form reemplazados por "Standard"

✅ 5. No se encontraron registros duplicados

✅ 6. Tipos de datos numéricos verificados


### 4.3 Estado DESPUÉS de la Limpieza

In [10]:
# ============================================================
# Estado del dataset DESPUÉS de la limpieza
# ============================================================
print('=' * 60)
print('📋 ESTADO DESPUÉS DE LA LIMPIEZA')
print('=' * 60)

# Dimensiones
print(f'\n📐 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')

# Valores nulos
nulos_despues = df.isnull().sum()
print(f'\n🔍 Valores nulos: {nulos_despues.sum()}')

# Duplicados
print(f'🔍 Registros duplicados: {df.duplicated().sum()}')

# Columnas normalizadas
print(f'\n📋 Columnas normalizadas: {list(df.columns)}')

# Tipos de datos
print(f'\n📊 Tipos de datos:')
for col in df.columns:
    print(f'   {col:20s} → {df[col].dtype}')

# Resumen de tipos de Pokémon
print(f'\n🎯 Tipos únicos de Pokémon (Type1): {df["type1"].nunique()}')
print(f'🎯 Generaciones: {sorted(df["generation"].unique())}')

# Muestra final
print(f'\n📄 Muestra del dataset limpio:')
df.head()

📋 ESTADO DESPUÉS DE LA LIMPIEZA

📐 Dimensiones: 1215 filas × 13 columnas

🔍 Valores nulos: 0
🔍 Registros duplicados: 0

📋 Columnas normalizadas: ['id', 'name', 'form', 'type1', 'type2', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'generation']

📊 Tipos de datos:
   id                   → int64
   name                 → object
   form                 → object
   type1                → object
   type2                → object
   total                → int64
   hp                   → int64
   attack               → int64
   defense              → int64
   sp_atk               → int64
   sp_def               → int64
   speed                → int64
   generation           → int64

🎯 Tipos únicos de Pokémon (Type1): 18
🎯 Generaciones: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

📄 Muestra del dataset limpio:


,id,name,form,type1,type2,total,hp,attack,defense,sp_atk,sp_def,speed,generation
0,1,Bulbasaur,Standard,Grass,Poison,318,45,49,49,65,65,45,1
1,2,Ivysaur,Standard,Grass,Poison,405,60,62,63,80,80,60,1
2,3,Venusaur,Standard,Grass,Poison,525,80,82,83,100,100,80,1
3,4,Charmander,Standard,Fire,None,309,39,52,43,60,50,65,1
4,5,Charmeleon,Standard,Fire,None,405,58,64,58,80,65,80,1


### 4.4 Resumen Visual del Dataset Limpio

In [11]:
# ============================================================
# Distribución de Pokémon por Tipo Principal
# ============================================================
print('📊 Distribución de Pokémon por Tipo Principal:')
print('=' * 50)
tipo_counts = df['type1'].value_counts()
for tipo, count in tipo_counts.items():
    barra = '█' * (count // 5)
    print(f'   {tipo:12s} │ {barra} {count}')

print(f'\n📊 Distribución de Pokémon por Generación:')
print('=' * 50)
gen_counts = df['generation'].value_counts().sort_index()
for gen, count in gen_counts.items():
    barra = '█' * (count // 5)
    print(f'   Gen {int(gen):1d}      │ {barra} {count}')

📊 Distribución de Pokémon por Tipo Principal:
   Water        │ ██████████████████████████████ 150
   Normal       │ ██████████████████████████ 134
   Grass        │ ██████████████████████ 113
   Bug          │ ██████████████████ 91
   Psychic      │ ████████████████ 82
   Fire         │ ███████████████ 76
   Electric     │ ██████████████ 74
   Rock         │ █████████████ 68
   Dark         │ ███████████ 56
   Fighting     │ ██████████ 50
   Poison       │ █████████ 49
   Dragon       │ █████████ 49
   Ghost        │ █████████ 47
   Ground       │ █████████ 47
   Steel        │ █████████ 45
   Ice          │ ████████ 43
   Fairy        │ ██████ 31
   Flying       │ ██ 10

📊 Distribución de Pokémon por Generación:
   Gen 1      │ ██████████████████████████████ 151
   Gen 2      │ ████████████████████ 100
   Gen 3      │ ████████████████████████████ 141
   Gen 4      │ ███████████████████████ 118
   Gen 5      │ █████████████████████████████████ 165
   Gen 6      │ █████████████████████

---

## 5. Selección y Justificación de Variables Estadísticas

Para el análisis y la visualización 3D, seleccionamos las **6 estadísticas base** que definen las capacidades de combate de cada Pokémon:

| Variable | Descripción | Justificación |
|----------|-------------|---------------|
| `hp` | Puntos de Vida | Determina la resistencia total del Pokémon en combate |
| `attack` | Ataque Físico | Mide el poder de los movimientos físicos |
| `defense` | Defensa Física | Capacidad de resistir ataques físicos |
| `sp_atk` | Ataque Especial | Poder de los movimientos especiales (fuego, agua, etc.) |
| `sp_def` | Defensa Especial | Capacidad de resistir ataques especiales |
| `speed` | Velocidad | Determina el orden de turno en combate |

Estas 6 estadísticas son el estándar oficial de la franquicia Pokémon y permiten crear un **perfil estadístico completo** de cada Pokémon, identificando roles como tanques (alta defensa), sweepers (alto ataque/velocidad) o soportes (alta defensa especial/HP).

In [12]:
# ============================================================
# Selección de variables estadísticas para el análisis
# ============================================================
stats_cols = ['hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed']

print('📊 Variables estadísticas seleccionadas para el análisis:')
print('=' * 60)
for i, col in enumerate(stats_cols, 1):
    min_val = df[col].min()
    max_val = df[col].max()
    mean_val = df[col].mean()
    print(f'   {i}. {col:10s} │ Mín: {min_val:5.0f} │ Máx: {max_val:5.0f} │ Media: {mean_val:6.1f}')

print(f'\n📋 Total de variables seleccionadas: {len(stats_cols)}')
print(f'📋 Variable adicional disponible: "total" (suma de las 6 estadísticas)')

📊 Variables estadísticas seleccionadas para el análisis:
   1. hp         │ Mín:     1 │ Máx:   255 │ Media:   71.2
   2. attack     │ Mín:     5 │ Máx:   190 │ Media:   81.2
   3. defense    │ Mín:     5 │ Máx:   250 │ Media:   75.0
   4. sp_atk     │ Mín:    10 │ Máx:   194 │ Media:   73.2
   5. sp_def     │ Mín:    20 │ Máx:   250 │ Media:   72.4
   6. speed      │ Mín:     5 │ Máx:   200 │ Media:   70.0

📋 Total de variables seleccionadas: 6
📋 Variable adicional disponible: "total" (suma de las 6 estadísticas)


---

## 6. Creación de la Columna `promedio_estadisticas`

Calculamos una nueva columna que representa el **promedio aritmético** de las 6 estadísticas base. Esta métrica unificada permite comparar Pokémon de forma directa y será utilizada como uno de los ejes del gráfico 3D.

In [13]:
# ============================================================
# Creación de la columna promedio_estadisticas
# ============================================================
df['promedio_estadisticas'] = df[stats_cols].mean(axis=1).round(2)

print('✅ Columna "promedio_estadisticas" creada exitosamente')
print(f'\n📊 Estadísticas del promedio:')
print(f'   Mínimo:  {df["promedio_estadisticas"].min():.2f}')
print(f'   Máximo:  {df["promedio_estadisticas"].max():.2f}')
print(f'   Media:   {df["promedio_estadisticas"].mean():.2f}')
print(f'   Mediana: {df["promedio_estadisticas"].median():.2f}')
print(f'   Std:     {df["promedio_estadisticas"].std():.2f}')

# Top 10 Pokémon con mayor promedio
print(f'\n🏆 Top 10 Pokémon con mayor promedio de estadísticas:')
print('=' * 60)
top10 = df.nlargest(10, 'promedio_estadisticas')[['id', 'name', 'form', 'type1', 'generation', 'promedio_estadisticas']]
for idx, row in top10.iterrows():
    print(f'   #{row["id"]:4.0f} {row["name"]:20s} ({row["form"]:15s}) │ {row["type1"]:10s} │ Gen {row["generation"]:.0f} │ Prom: {row["promedio_estadisticas"]:.2f}')

# Bottom 5
print(f'\n📉 Top 5 Pokémon con menor promedio de estadísticas:')
bottom5 = df.nsmallest(5, 'promedio_estadisticas')[['id', 'name', 'type1', 'generation', 'promedio_estadisticas']]
for idx, row in bottom5.iterrows():
    print(f'   #{row["id"]:4.0f} {row["name"]:20s} │ {row["type1"]:10s} │ Gen {row["generation"]:.0f} │ Prom: {row["promedio_estadisticas"]:.2f}')

✅ Columna "promedio_estadisticas" creada exitosamente

📊 Estadísticas del promedio:
   Mínimo:  29.17
   Máximo:  187.50
   Media:   73.85
   Mediana: 77.50
   Std:     20.20

🏆 Top 10 Pokémon con mayor promedio de estadísticas:
   # 890 Eternatus            (Eternamax      ) │ Poison     │ Gen 8 │ Prom: 187.50
   # 150 Mewtwo               (Mega Mewtwo X  ) │ Psychic    │ Gen 6 │ Prom: 130.00
   # 150 Mewtwo               (Mega Mewtwo Y  ) │ Psychic    │ Gen 6 │ Prom: 130.00
   # 384 Rayquaza             (Mega Rayquaza  ) │ Dragon     │ Gen 6 │ Prom: 130.00
   # 382 Kyogre               (Primal Kyogre  ) │ Water      │ Gen 6 │ Prom: 128.33
   # 383 Groudon              (Primal Groudon ) │ Ground     │ Gen 6 │ Prom: 128.33
   # 800 Necrozma             (Ultra Necrozma ) │ Psychic    │ Gen 7 │ Prom: 125.67
   # 493 Arceus               (Standard       ) │ Normal     │ Gen 4 │ Prom: 120.00
   # 718 Zygarde              (Complete Forme ) │ Dragon     │ Gen 7 │ Prom: 118.00
   # 646 Kyurem

---

## 7. Análisis Estadístico Descriptivo

Realizamos un análisis estadístico completo que incluye **media, mediana, mínimo, máximo y desviación estándar** de las estadísticas seleccionadas, segmentado por tipo principal y por generación.

### 7.1 Estadísticas Descriptivas Generales

In [14]:
# ============================================================
# Estadísticas descriptivas generales
# ============================================================
stats_resumen = df[stats_cols + ['promedio_estadisticas']].agg(
    ['mean', 'median', 'min', 'max', 'std']
).round(2)

stats_resumen.index = ['Media', 'Mediana', 'Mínimo', 'Máximo', 'Desv. Estándar']

print('📊 Estadísticas Descriptivas Generales:')
print('=' * 80)
display(stats_resumen)

📊 Estadísticas Descriptivas Generales:


,hp,attack,defense,sp_atk,sp_def,speed,promedio_estadisticas
Media,71.24,81.15,75.01,73.22,72.44,70.03,73.85
Mediana,70.00,80.00,70.00,65.00,70.00,68.00,77.50
Mínimo,1.00,5.00,5.00,10.00,20.00,5.00,29.17
Máximo,255.00,190.00,250.00,194.00,250.00,200.00,187.50
Desv. Estándar,26.93,32.04,30.74,32.76,27.58,30.16,20.20


### 7.2 Estadísticas por Tipo Principal

In [15]:
# ============================================================
# Promedio de estadísticas por Tipo Principal
# ============================================================
stats_por_tipo = df.groupby('type1')[stats_cols + ['promedio_estadisticas']].mean().round(2)
stats_por_tipo = stats_por_tipo.sort_values('promedio_estadisticas', ascending=False)

print('📊 Promedio de Estadísticas por Tipo Principal (ordenado por promedio):')
print('=' * 80)
display(stats_por_tipo)

📊 Promedio de Estadísticas por Tipo Principal (ordenado por promedio):


,hp,attack,defense,sp_atk,sp_def,speed,promedio_estadisticas
type1,,,,,,,
Dragon,84.57,103.82,80.82,90.12,83.57,84.65,87.93
Steel,71.18,92.51,114.27,75.16,79.22,57.56,81.65
Psychic,73.84,75.65,71.52,98.72,86.30,80.37,81.07
Fighting,75.74,104.96,76.44,55.16,69.66,76.12,76.35
Fire,70.76,84.47,69.32,86.83,71.64,73.96,76.16
Flying,70.90,81.90,67.40,72.60,70.90,86.80,75.08
Dark,72.89,85.48,70.98,72.27,70.55,77.54,74.95
Electric,63.84,73.15,65.84,88.68,70.41,87.50,74.90
Fairy,72.13,71.06,73.65,78.13,87.19,67.06,74.87


### 7.3 Estadísticas por Generación

In [16]:
# ============================================================
# Promedio de estadísticas por Generación
# ============================================================
stats_por_gen = df.groupby('generation')[stats_cols + ['promedio_estadisticas']].agg(
    ['mean', 'median', 'std']
).round(2)

# Simplificar para mostrar solo el promedio_estadisticas por generación
resumen_gen = df.groupby('generation').agg(
    total_pokemon=('id', 'count'),
    prom_hp=('hp', 'mean'),
    prom_attack=('attack', 'mean'),
    prom_defense=('defense', 'mean'),
    prom_sp_atk=('sp_atk', 'mean'),
    prom_sp_def=('sp_def', 'mean'),
    prom_speed=('speed', 'mean'),
    prom_general=('promedio_estadisticas', 'mean'),
    std_general=('promedio_estadisticas', 'std')
).round(2)

print('📊 Resumen Estadístico por Generación:')
print('=' * 100)
display(resumen_gen)

📊 Resumen Estadístico por Generación:


,total_pokemon,prom_hp,prom_attack,prom_defense,prom_sp_atk,prom_sp_def,prom_speed,prom_general,std_general
generation,,,,,,,,,
1,151,64.21,72.91,68.23,67.14,66.09,69.07,67.94,16.65
2,100,70.98,68.26,69.69,64.50,72.34,61.41,67.86,18.74
3,141,65.43,73.94,69.48,68.91,67.04,63.46,68.04,19.43
4,118,72.22,79.13,76.58,74.51,75.75,69.70,74.65,19.87
5,165,71.71,82.44,72.08,71.99,68.31,68.37,72.48,17.99
6,131,72.55,94.92,87.88,89.27,84.36,76.63,84.27,22.61
7,122,70.43,86.51,78.28,74.57,73.82,69.28,75.48,20.36
8,147,75.35,86.20,76.83,75.01,73.90,73.51,76.80,21.45
9,140,78.69,83.85,77.01,72.67,73.00,76.94,77.03,19.31


---

## 8. Preparación de Variables para la Visualización

Preparamos y ordenamos las variables de **generación** y **tipo principal** para su correcta representación en la visualización 3D. Asignamos una paleta de **colores oficiales** de los tipos Pokémon.

In [17]:
# ============================================================
# Preparación de variables de generación y tipo
# ============================================================

# 1. Asegurar que generation sea entero y esté ordenada
df['generation'] = df['generation'].astype(int)
generaciones_ordenadas = sorted(df['generation'].unique())
print(f'✅ Generaciones disponibles (ordenadas): {generaciones_ordenadas}')

# 2. Ordenar tipos principales y asignar código numérico para eje Y
tipos_ordenados = sorted(df['type1'].unique())
tipo_a_numero = {tipo: i for i, tipo in enumerate(tipos_ordenados)}
df['type1_num'] = df['type1'].map(tipo_a_numero)

print(f'\n✅ Tipos principales ({len(tipos_ordenados)} tipos):')
for tipo, num in tipo_a_numero.items():
    count = (df['type1'] == tipo).sum()
    print(f'   {num:2d}. {tipo:12s} → {count} Pokémon')

# 3. Paleta de colores oficial de tipos Pokémon
colores_tipo = {
    'Normal': '#A8A878',
    'Fire': '#F08030',
    'Water': '#6890F0',
    'Electric': '#F8D030',
    'Grass': '#78C850',
    'Ice': '#98D8D8',
    'Fighting': '#C03028',
    'Poison': '#A040A0',
    'Ground': '#E0C068',
    'Flying': '#A890F0',
    'Psychic': '#F85888',
    'Bug': '#A8B820',
    'Rock': '#B8A038',
    'Ghost': '#705898',
    'Dragon': '#7038F8',
    'Dark': '#705848',
    'Steel': '#B8B8D0',
    'Fairy': '#EE99AC'
}

# Asignar color a cada Pokémon según su tipo
df['color_tipo'] = df['type1'].map(colores_tipo)

print(f'\n✅ Paleta de colores asignada ({len(colores_tipo)} tipos con color)')
print(f'   Pokémon sin color asignado: {df["color_tipo"].isna().sum()}')

✅ Generaciones disponibles (ordenadas): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

✅ Tipos principales (18 tipos):
    0. Bug          → 91 Pokémon
    1. Dark         → 56 Pokémon
    2. Dragon       → 49 Pokémon
    3. Electric     → 74 Pokémon
    4. Fairy        → 31 Pokémon
    5. Fighting     → 50 Pokémon
    6. Fire         → 76 Pokémon
    7. Flying       → 10 Pokémon
    8. Ghost        → 47 Pokémon
    9. Grass        → 113 Pokémon
   10. Ground       → 47 Pokémon
   11. Ice          → 43 Pokémon
   12. Normal       → 134 Pokémon
   13. Poison       → 49 Pokémon
   14. Psychic      → 82 Pokémon
   15. Rock         → 68 Pokémon
   16. Steel        → 45 Pokémon
   17. Water        → 150 Pokémon

✅ Paleta de colores asignada (18 tipos con color)
   Pokémon sin color asignado: 0


---

## 9. Obtención y Validación de Sprites de Pokémon

Los sprites (imágenes pequeñas) de cada Pokémon se obtienen del repositorio de [PokeAPI Sprites](https://github.com/PokeAPI/sprites) en GitHub. La URL de cada sprite se construye a partir del **ID del Pokédex Nacional**:

```
https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{id}.png
```

Esto nos permite integrar las imágenes directamente en la visualización interactiva sin necesidad de descargarlas localmente.

In [18]:
# ============================================================
# Construcción de URLs de sprites para cada Pokémon
# ============================================================
base_url_sprite = 'https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/'

# Construir la URL del sprite basada en el ID del Pokédex
df['sprite_url'] = df['id'].astype(int).apply(lambda x: f'{base_url_sprite}{x}.png')

print('✅ URLs de sprites construidas para cada Pokémon')
print(f'   Total de URLs generadas: {len(df)}')
print(f'\n📋 Ejemplos de URLs de sprites:')
for _, row in df.head(5).iterrows():
    print(f'   #{row["id"]:4.0f} {row["name"]:15s} → {row["sprite_url"]}')

✅ URLs de sprites construidas para cada Pokémon
   Total de URLs generadas: 1215

📋 Ejemplos de URLs de sprites:
   #   1 Bulbasaur       → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/1.png
   #   2 Ivysaur         → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/2.png
   #   3 Venusaur        → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/3.png
   #   4 Charmander      → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/4.png
   #   5 Charmeleon      → https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/5.png


In [19]:
# ============================================================
# Validación de sprites con una muestra representativa
# ============================================================
import urllib.request

# Verificar sprites de una muestra (1 por generación)
print('🔍 Verificación de sprites (muestra por generación):')
print('=' * 60)

muestra = df.drop_duplicates(subset='generation').sort_values('generation')
sprites_validos = 0
sprites_totales = 0

for _, row in muestra.iterrows():
    sprites_totales += 1
    try:
        req = urllib.request.Request(row['sprite_url'], method='HEAD')
        response = urllib.request.urlopen(req, timeout=5)
        status = response.getcode()
        if status == 200:
            sprites_validos += 1
            print(f'   ✅ Gen {row["generation"]} │ #{row["id"]:4.0f} {row["name"]:15s} │ Sprite disponible')
        else:
            print(f'   ⚠️ Gen {row["generation"]} │ #{row["id"]:4.0f} {row["name"]:15s} │ Status: {status}')
    except Exception as e:
        print(f'   ❌ Gen {row["generation"]} │ #{row["id"]:4.0f} {row["name"]:15s} │ Error: {str(e)[:40]}')

print(f'\n📊 Resultado: {sprites_validos}/{sprites_totales} sprites verificados correctamente')

🔍 Verificación de sprites (muestra por generación):


   ✅ Gen 1 │ #   1 Bulbasaur       │ Sprite disponible


   ✅ Gen 2 │ # 152 Chikorita       │ Sprite disponible


   ✅ Gen 3 │ # 252 Treecko         │ Sprite disponible


   ✅ Gen 4 │ # 387 Turtwig         │ Sprite disponible


   ✅ Gen 5 │ # 494 Victini         │ Sprite disponible


   ✅ Gen 6 │ #   3 Venusaur        │ Sprite disponible


   ✅ Gen 7 │ #  19 Rattata         │ Sprite disponible


   ✅ Gen 8 │ #  52 Meowth          │ Sprite disponible


   ✅ Gen 9 │ # 128 Tauros          │ Sprite disponible

📊 Resultado: 9/9 sprites verificados correctamente


In [20]:
# ============================================================
# Vista previa de sprites (primeros 5 Pokémon)
# ============================================================
print('🖼️ Vista previa de sprites (Pokémon iniciales):')
print('=' * 60)

html_preview = '<div style="display: flex; gap: 20px; align-items: center; flex-wrap: wrap; background: #1a1a2e; padding: 20px; border-radius: 10px;">'
for _, row in df.head(9).iterrows():
    html_preview += f'''
    <div style="text-align: center; color: white;">
        <img src="{row['sprite_url']}" width="80" height="80" style="image-rendering: pixelated;">
        <br><span style="font-size: 11px;">#{int(row['id'])} {row['name']}</span>
        <br><span style="font-size: 10px; color: {row['color_tipo']};">{row['type1']}</span>
    </div>'''
html_preview += '</div>'

display(HTML(html_preview))

🖼️ Vista previa de sprites (Pokémon iniciales):


---

### ✅ Resumen del Dataset Preparado

El dataset está listo para la visualización 3D con las siguientes columnas adicionales:
- `promedio_estadisticas` — Promedio de las 6 estadísticas base
- `type1_num` — Codificación numérica del tipo principal
- `color_tipo` — Color hexadecimal asociado al tipo
- `sprite_url` — URL del sprite oficial del Pokémon

In [21]:
# ============================================================
# Resumen final del dataset preparado
# ============================================================
print('📋 RESUMEN DEL DATASET PREPARADO PARA VISUALIZACIÓN 3D')
print('=' * 60)
print(f'   📊 Total de registros:    {len(df)}')
print(f'   📋 Total de columnas:     {len(df.columns)}')
print(f'   🎮 Generaciones:          {df["generation"].nunique()} ({df["generation"].min()}-{df["generation"].max()})')
print(f'   🎯 Tipos principales:     {df["type1"].nunique()}')
print(f'   📈 Rango de promedios:    {df["promedio_estadisticas"].min():.1f} - {df["promedio_estadisticas"].max():.1f}')
print(f'   🖼️ Sprites configurados:  {df["sprite_url"].notna().sum()}')
print(f'\n📋 Columnas del dataset final:')
for i, col in enumerate(df.columns, 1):
    print(f'   {i:2d}. {col}')

df.head(3)

📋 RESUMEN DEL DATASET PREPARADO PARA VISUALIZACIÓN 3D
   📊 Total de registros:    1215
   📋 Total de columnas:     17
   🎮 Generaciones:          9 (1-9)
   🎯 Tipos principales:     18
   📈 Rango de promedios:    29.2 - 187.5
   🖼️ Sprites configurados:  1215

📋 Columnas del dataset final:
    1. id
    2. name
    3. form
    4. type1
    5. type2
    6. total
    7. hp
    8. attack
    9. defense
   10. sp_atk
   11. sp_def
   12. speed
   13. generation
   14. promedio_estadisticas
   15. type1_num
   16. color_tipo
   17. sprite_url


,id,name,form,type1,type2,total,hp,attack,defense,sp_atk,sp_def,speed,generation,promedio_estadisticas,type1_num,color_tipo,sprite_url
0,1,Bulbasaur,Standard,Grass,Poison,318,45,49,49,65,65,45,1,53.0,9,#78C850,https://raw.githubusercontent.com/PokeAPI/spri...
1,2,Ivysaur,Standard,Grass,Poison,405,60,62,63,80,80,60,1,67.5,9,#78C850,https://raw.githubusercontent.com/PokeAPI/spri...
2,3,Venusaur,Standard,Grass,Poison,525,80,82,83,100,100,80,1,87.5,9,#78C850,https://raw.githubusercontent.com/PokeAPI/spri...
